In [ ]:
from google.colab import files
import pdfplumber
import spacy

In [ ]:
!python -m spacy download en_core_web_sm
!pip install pdfplumber
!pip install sentence-transformers
!pip install reportlab

In [28]:
print("Upload company policy PDF (optional)")
custom_uploaded = files.upload()

Upload company policy PDF (optional)


Saving company_policy.pdf to company_policy.pdf


In [45]:
import spacy

# Load model
nlp = spacy.load("en_core_web_sm")

In [29]:
custom_text = ""

if len(custom_uploaded) > 0:
    file_name = list(custom_uploaded.keys())[0]

    with pdfplumber.open(file_name) as pdf:
        for page in pdf.pages:
            custom_text += (page.extract_text() or "") + "\n"

In [31]:
custom_clauses = []

if custom_text:
    doc = nlp(custom_text)
    custom_clauses = [sent.text.strip() for sent in doc.sents]

In [32]:
custom_rules = []

for i, clause in enumerate(custom_clauses):
    custom_rules.append({
        "rule_id": f"custom_{i}",
        "law": "Company Policy",
        "description": clause,
        "keywords": [],
        "required_terms": [],
        "severity": "Medium"
    })

In [33]:
import json

rules = [
    {
        "rule_id": 1,
        "law": "Contract Basics",
        "description": "A contract should clearly identify all parties involved, including their names and roles.",
        "keywords": ["agreement", "company", "party"],
        "required_terms": ["between", "and"],
        "severity": "High"
    },
    {
        "rule_id": 2,
        "law": "Contract Basics",
        "description": "A contract should clearly specify the effective date when obligations begin.",
        "keywords": ["agreement", "effective"],
        "required_terms": ["date", "effective"],
        "severity": "High"
    },
    {
        "rule_id": 3,
        "law": "Contract Law",
        "description": "A contract should define how and under what conditions it can be terminated.",
        "keywords": ["terminate", "termination"],
        "required_terms": ["notice", "days"],
        "severity": "High"
    }
]

with open("rules.json", "w") as f:
    json.dump(rules, f, indent=4)

print("rules.json created!")

rules.json created!


In [34]:
with open("rules.json") as f:
    base_rules = json.load(f)

all_rules = base_rules + custom_rules

In [35]:
from google.colab import files
uploaded = files.upload()

Saving fake_contract.pdf to fake_contract (1).pdf


In [36]:
!pip install pdfplumber

In [37]:
import pdfplumber

# Get file name
file_name = list(uploaded.keys())[0]

text = ""

with pdfplumber.open(file_name) as pdf:
    for page in pdf.pages:
        text += (page.extract_text() or "") + "\n"

print(text[:1000])  # preview first 1000 chars

This Agreement is entered into between ABC Corp and XYZ Ltd.
This Agreement shall be effective from January 1, 2024.
The company may share user data with third parties.
Either party may terminate this Agreement.
The company shall ensure secure handling of user data.
All disputes shall be governed by applicable laws.



In [38]:
doc = nlp(text)
clauses = [sent.text.strip() for sent in doc.sents if sent.text.strip()]

# Print clauses
for i, clause in enumerate(clauses):
    print(f"{i+1}. {clause}")

1. This Agreement is entered into between ABC Corp and XYZ Ltd.
This Agreement shall be effective from January 1, 2024.
2. The company may share user data with third parties.
3. Either party may terminate this Agreement.
4. The company shall ensure secure handling of user data.
5. All disputes shall be governed by applicable laws.


In [39]:
!pip install sentence-transformers

In [40]:
from sentence_transformers import SentenceTransformer, util
import json
import re

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract rule descriptions
rule_texts = [rule["description"] for rule in all_rules]

# Convert rules to embeddings
rule_embeddings = model.encode(rule_texts, convert_to_tensor=True)


# 🔥 Helper: detect date
def contains_date(text):
    date_patterns = [
        r'\b\d{1,2}/\d{1,2}/\d{2,4}\b',
        r'\b(January|February|March|April|May|June|July|August|September|October|November|December)\b'
    ]

    for pattern in date_patterns:
        if re.search(pattern, text, re.IGNORECASE):
            return True
    return False

# 🔥 Main violation check
def check_violation(clause, rule):
    clause_lower = clause.lower()
    missing_terms = []

    for term in rule["required_terms"]:

        if term == "date":
            if not contains_date(clause):
                missing_terms.append(term)
        else:
            if term not in clause_lower:
                missing_terms.append(term)

    return len(missing_terms) > 0, missing_terms


# 🔥 MAIN LOOP
results = []

for clause in clauses:
    clause_embedding = model.encode(clause, convert_to_tensor=True)
    scores = util.cos_sim(clause_embedding, rule_embeddings)[0]

    best_match_idx = scores.argmax()
    best_score = scores[best_match_idx].item()

    matched_rule = all_rules[best_match_idx]

    is_violation, missing = check_violation(clause, matched_rule)

    # 🔥 Confidence (based on similarity)
    confidence = round(best_score, 2)

    # 🔥 Explanation
    if is_violation:
        explanation = f"This clause relates to '{matched_rule['law']}' but is missing required elements: {', '.join(missing)}."
    else:
        explanation = f"This clause satisfies the requirements of '{matched_rule['law']}'."

    result = {
        "clause": clause,
        "matched_rule": matched_rule["description"],
        "law": matched_rule["law"],
        "severity": matched_rule["severity"],
        "confidence": confidence,
        "risk": is_violation,
        "explanation": explanation
    }

    results.append(result)

# Print nicely
import json
print(json.dumps(results, indent=4))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[
    {
        "clause": "This Agreement is entered into between ABC Corp and XYZ Ltd.",
        "matched_rule": "A contract should clearly identify all parties involved, including their names and roles.",
        "law": "Contract Basics",
        "severity": "High",
        "confidence": 0.34,
        "risk": false,
        "explanation": "This clause satisfies the requirements of 'Contract Basics'."
    },
    {
        "clause": "This Agreement shall be effective from January 1, 2024.",
        "matched_rule": "A contract should clearly specify the effective date when obligations begin.",
        "law": "Contract Basics",
        "severity": "High",
        "confidence": 0.47,
        "risk": false,
        "explanation": "This clause satisfies the requirements of 'Contract Basics'."
    },
    {
        "clause": "Either party may terminate this Agreement with 30 days notice.",
        "matched_rule": "A contract should define how and under what conditions it can be terminated.",


In [41]:
def generate_report(results):
    report = "\n COMPLIANCE AUDIT REPORT\n"
    report += "="*50 + "\n\n"

    total = len(results)
    risks = [r for r in results if r["risk"]]

    report += f"Total Clauses Analyzed: {total}\n"
    report += f"Total Risks Found: {len(risks)}\n\n"

    report += "⚠️ RISK DETAILS:\n"
    report += "-"*50 + "\n"

    for r in risks:
        report += f"\nClause: {r['clause']}\n"
        report += f"Law: {r['law']}\n"
        report += f"Severity: {r['severity']}\n"
        report += f"Explanation: {r['explanation']}\n"
        report += "-"*50 + "\n"

    report += "\n✅ SAFE CLAUSES:\n"
    report += "-"*50 + "\n"

    for r in results:
        if not r["risk"]:
            report += f"- {r['clause']}\n"

    return report

In [46]:
report = generate_report(results)

In [47]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

def save_report_as_pdf(report_text, filename="audit_report.pdf"):
    doc = SimpleDocTemplate(filename)
    styles = getSampleStyleSheet()

    content = []

    for line in report_text.split("\n"):
        content.append(Paragraph(line, styles["Normal"]))
        content.append(Spacer(1, 10))

    doc.build(content)

    print(f"Report saved as {filename}")

In [48]:
save_report_as_pdf(report)

Report saved as audit_report.pdf
